In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score

In [3]:
# Dataset can be downloaded here:
# https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
df = pd.read_csv('course_lead_scoring_2026.csv')
df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [4]:
### Data preparation
# Check for missing values. Categorical features -> fill with 'NA', numerical features -> fill with 0.0.

categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']

print(df.isnull().sum())

for c in categorical:
    df[c] = df[c].fillna('NA')
for c in numerical:
    df[c] = df[c].fillna(0.0)

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64


In [5]:
# What is the most frequent observation (mode) for the column `industry`?
print(df['industry'].mode()[0])

technology


In [6]:
# Create the correlation matrix for the numerical features. Which two features have the biggest correlation?
corr = df[numerical].corr()
print(corr)

pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count'),
]
for a, b in pairs:
    print(a, '/', b, '->', round(corr.loc[a, b], 4))

                          annual_income  number_of_courses_viewed  \
annual_income                  1.000000                  0.161300   
number_of_courses_viewed       0.161300                  1.000000   
interaction_count              0.122842                  0.721609   
lead_score                     0.229496                  0.757204   

                          interaction_count  lead_score  
annual_income                      0.122842    0.229496  
number_of_courses_viewed           0.721609    0.757204  
interaction_count                  1.000000    0.915746  
lead_score                         0.915746    1.000000  
interaction_count / lead_score -> 0.9157
number_of_courses_viewed / lead_score -> 0.7572
number_of_courses_viewed / interaction_count -> 0.7216
annual_income / interaction_count -> 0.1228


In [7]:
# Split the data
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

# Make sure the target is not in the feature dataframes
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [9]:
# Mutual information score between converted and the categorical variables (training set only), rounded to 2 decimals. Which one is the biggest?
for c in categorical:
    print(c, round(mutual_info_score(df_train[c], y_train), 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


In [13]:
#Train a logistic regression with one-hot encoding of the categorical variables. Accuracy on validation, rounded to 2 decimals.

features = categorical + numerical


def train_model(df_tr, y_tr, cols, C=1.0):
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(df_tr[cols].to_dict(orient='records'))
    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X, y_tr)
    return dv, model


def evaluate(dv, model, df_v, y_v, cols):
    X = dv.transform(df_v[cols].to_dict(orient='records'))
    return accuracy_score(y_v, model.predict(X))

dv, model = train_model(df_train, y_train, features)
accuracy = evaluate(dv, model, df_val, y_val, features)
print('accuracy =', round(accuracy, 2))

accuracy = 0.65


In [14]:
# Feature elimination: remove each feature, retrain, and look at the drop in accuracy vs the full model. Which feature has the smallest difference?

for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    subset = [c for c in features if c != f]
    dv_s, model_s = train_model(df_train, y_train, subset)
    acc_s = evaluate(dv_s, model_s, df_val, y_val, subset)
    print('without', f, '-> diff =', round(accuracy - acc_s, 5))

without lead_source -> diff = 0.003
without number_of_courses_viewed -> diff = 0.002
without interaction_count -> diff = 0.044


In [15]:
# Regularized logistic regression. Try C in [0.000001, 0.00001, 0.0001, 0.001]. Accuracy on validation rounded to 3 decimals. On ties, pick the smallest C.

for C in [0.000001, 0.00001, 0.0001, 0.001]:
    dv_c, model_c = train_model(df_train, y_train, features, C=C)
    acc_c = evaluate(dv_c, model_c, df_val, y_val, features)
    print('C =', C, '-> accuracy =', round(acc_c, 3))
# Answer Q6: 0.001 (best accuracy 0.645)

C = 1e-06 -> accuracy = 0.598
C = 1e-05 -> accuracy = 0.598
C = 0.0001 -> accuracy = 0.613
C = 0.001 -> accuracy = 0.645
